# SegRNN — one-notebook reproduction (reconstruction + improvement)

Runtime → Change runtime type → **T4 GPU** before running. Dataset CSVs must
already be in Google Drive at `MyDrive/ts-project/dataset/`.

**Run every cell top to bottom, once.** Nothing needs to be copied out and
pasted back anywhere — every table and graph is produced and shown inline,
right here.

What this notebook does, in order:
- **Part 1** — trains and evaluates the original **SegRNN** (the paper
  reconstruction) on ETTh1, all four horizons.
- **Part 2** — computes the classical baselines (naive, seasonal-naive) on
  the identical data pipeline, deterministic, no GPU needed.
- **Part 3** — table + graphs: paper vs. reconstruction vs. baselines.
- **Part 4** — trains and evaluates **SegRNNTime**, the Stage 2 improved
  architecture (calendar features wired into the encoder).
- **Part 5** — final table + graphs (new figures): paper vs. reconstruction
  vs. baselines vs. improved.
- **Optional** — save results/figures back into the repo and push.

Expect roughly 20–30 minutes total on a T4 (Parts 1 and 4 each train 4
models with early stopping; Parts 2/3/5 take seconds).

The actual method code lives in the repo as usual —
[`models/SegRNN.py`](../models/SegRNN.py),
[`models/SegRNNTime.py`](../models/SegRNNTime.py),
[`run_longExp.py`](../run_longExp.py),
[`scripts/baselines.py`](../scripts/baselines.py) — this notebook just
calls it and organizes the output into one place.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
REPO = "https://github.com/amitzr/SegRNN.git"
if not os.path.exists('/content/proj'):
    !git clone $REPO /content/proj
%cd /content/proj
!git pull

In [ ]:
import os
if not os.path.exists('/content/proj/dataset'):
    os.symlink('/content/drive/MyDrive/ts-project/dataset', '/content/proj/dataset')
!ls -la /content/proj/dataset | head
!pip install -q -r requirements.txt

## Setup: shared constants, training runner, plotting

Run this once. `run_horizon` launches `run_longExp.py` as a subprocess (the
exact same entry point the paper's own scripts use), streams its output live
so you can watch training progress, and parses the final `mse:.., mae:..`
line it prints — no manual copying, no stale results.

In [ ]:
import os, sys, re, csv, subprocess, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.getcwd())  # make scripts.*, utils.*, data_provider.* importable

HORIZONS = [96, 192, 336, 720]

# Paper's Table II, ETTh1, multivariate, L=720 (docs/SegRNN_paper.pdf)
PAPER = {
    'mse': {96: 0.351, 192: 0.392, 336: 0.423, 720: 0.466},
    'mae': {96: 0.392, 192: 0.414, 336: 0.433, 720: 0.472},
}

os.makedirs('results/figures', exist_ok=True)


def run_horizon(model, pred_len, mark_dim=None):
    """Launch run_longExp.py for one (model, horizon), stream its output
    live, and parse the final 'mse:X, mae:Y' line it prints.
    Returns (mse, mae)."""
    model_id = f'ETTh1_720_{pred_len}'
    cmd = [
        'python', '-u', 'run_longExp.py',
        '--is_training', '1', '--model_id', model_id, '--model', model, '--data', 'ETTh1',
        '--root_path', './dataset/', '--data_path', 'ETTh1.csv',
        '--features', 'M', '--seq_len', '720', '--pred_len', str(pred_len),
        '--seg_len', '24', '--enc_in', '7', '--d_model', '512',
        '--dropout', '0.1', '--rnn_type', 'gru', '--dec_way', 'pmf', '--channel_id', '1',
        '--train_epochs', '30', '--patience', '5',
        '--itr', '1', '--batch_size', '64', '--learning_rate', '0.0003',
    ]
    if mark_dim is not None:
        cmd += ['--mark_dim', str(mark_dim)]

    print(f'\n{"="*70}\n{model}  H={pred_len}\n{"="*70}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 5:
            tail.pop(0)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'{model} H={pred_len} failed (exit {proc.returncode}) -- see output above')

    m = re.search(r'mse:([\d.]+), mae:([\d.]+)', ''.join(tail))
    if not m:
        raise RuntimeError(f'Could not find mse/mae in output for {model} H={pred_len}')
    return float(m.group(1)), float(m.group(2))


# dataviz-validated categorical palette, fixed order
COLORS = {
    'Paper': '#2a78d6', 'Reconstruction': '#008300',
    'Naive': '#e87ba4', 'Seasonal-naive': '#eda100', 'Improved': '#1baf7a',
}
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#898781'
GRIDLINE, BASELINE_AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'


def plot_metric(metric_name, series, save_path=None):
    """series: list of (label, {horizon: value}), in display order.
    Always creates a brand-new figure."""
    n_series = len(series)
    x = np.arange(len(HORIZONS))
    group_width = 0.8
    bar_width = group_width / n_series

    fig, ax = plt.subplots(figsize=(9, 5.5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for i, (label, values) in enumerate(series):
        offsets = x - group_width / 2 + bar_width * (i + 0.5)
        heights = [values[h] for h in HORIZONS]
        bars = ax.bar(offsets, heights, width=bar_width * 0.9, color=COLORS[label],
                       label=label, edgecolor=SURFACE, linewidth=0.5)
        for bar, h in zip(bars, heights):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{h:.3f}',
                     ha='center', va='bottom', fontsize=7.5, color=INK_PRIMARY)

    ax.set_xticks(x)
    ax.set_xticklabels([f'H={h}' for h in HORIZONS], color=INK_SECONDARY)
    ax.set_ylabel(metric_name.upper(), color=INK_SECONDARY)
    ax.set_title(f'SegRNN on ETTh1 — {metric_name.upper()}', color=INK_PRIMARY, fontsize=13, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 1.14),
              ncol=n_series, fontsize=9, labelcolor=INK_SECONDARY)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, facecolor=SURFACE)
        print(f'saved {save_path}')
    plt.show()


def make_table(metric_name, series):
    rows = []
    for h in HORIZONS:
        row = {'Horizon': h}
        for label, values in series:
            row[label] = round(values[h], 4)
        rows.append(row)
    print(f'\n{metric_name.upper()}')
    return pd.DataFrame(rows).set_index('Horizon')

In [ ]:
from scripts.baselines import build_dataset, windowed_forecasts, seasonal_naive_scale, mase as mase_fn, infer_period
from utils.metrics import MSE, MAE
import argparse as _argparse

def run_baseline_horizon(pred_len):
    """Naive + seasonal-naive on the exact same Dataset/split/scaling as
    run_longExp.py (reuses scripts/baselines.py directly, no duplication)."""
    args = _argparse.Namespace(
        data='ETTh1', root_path='./dataset/', data_path='ETTh1.csv',
        seq_len=720, pred_len=pred_len, features='M', target='OT', freq='h',
        period=None,
    )
    period = infer_period(args)
    train_ds = build_dataset(args, 'train')
    test_ds = build_dataset(args, 'test')
    scale = seasonal_naive_scale(train_ds.data_x, period)
    trues, naive_preds, seasonal_preds = windowed_forecasts(test_ds.data_x, args.seq_len, args.pred_len, period)

    out = {}
    for name, preds in [('naive', naive_preds), ('seasonal_naive', seasonal_preds)]:
        out[name] = {'mse': MSE(preds, trues), 'mae': MAE(preds, trues), 'mase': mase_fn(preds, trues, scale)}
    return out

## Part 1 — Reconstruction (SegRNN)

Trains the paper's own reference configuration
(`scripts/SegRNN/etth1.sh`'s hyperparameters: `seq_len=720`, `seg_len=24`,
`d_model=512`, GRU, PMF decode, channel id on) for all four horizons.

In [ ]:
recon = {}
for h in HORIZONS:
    recon[h] = run_horizon('SegRNN', h)
print('\nReconstruction (SegRNN) done:', recon)

## Part 2 — Classical baselines (naive, seasonal-naive)

Deterministic, no GPU, seconds to run.

In [ ]:
baselines = {}
for h in HORIZONS:
    baselines[h] = run_baseline_horizon(h)
print('Baselines done:', baselines)

## Part 3 — Stage 1 results: paper vs. reconstruction vs. baselines

In [ ]:
def series_for(metric_name, include_improved=False):
    s = [
        ('Paper', PAPER[metric_name]),
        ('Reconstruction', {h: recon[h][0 if metric_name == 'mse' else 1] for h in HORIZONS}),
        ('Naive', {h: baselines[h]['naive'][metric_name] for h in HORIZONS}),
        ('Seasonal-naive', {h: baselines[h]['seasonal_naive'][metric_name] for h in HORIZONS}),
    ]
    if include_improved:
        s.append(('Improved', {h: improved[h][0 if metric_name == 'mse' else 1] for h in HORIZONS}))
    return s

mse_series = series_for('mse')
mae_series = series_for('mae')

display(make_table('mse', mse_series))
display(make_table('mae', mae_series))

plot_metric('mse', mse_series, save_path='results/figures/mse_comparison.png')
plot_metric('mae', mae_series, save_path='results/figures/mae_comparison.png')

## Part 4 — Improved architecture (SegRNNTime)

`models/SegRNNTime.py` wires the calendar features (hour-of-day,
day-of-week, day-of-month, day-of-year) that the data loader already
computes but the original SegRNN discards, into the encoder: each input
segment's raw values are concatenated with that segment's **last-timestep**
calendar feature vector before the `Linear(w→d)` embedding. Same
hyperparameters, split, and seed as Part 1 — only the architecture and
`--mark_dim 4` differ, so the comparison is apples-to-apples.

In [ ]:
improved = {}
for h in HORIZONS:
    improved[h] = run_horizon('SegRNNTime', h, mark_dim=4)
print('\nImproved (SegRNNTime) done:', improved)

## Part 5 — Final results: paper vs. reconstruction vs. baselines vs. improved

In [ ]:
mse_series_final = series_for('mse', include_improved=True)
mae_series_final = series_for('mae', include_improved=True)

display(make_table('mse', mse_series_final))
display(make_table('mae', mae_series_final))

plot_metric('mse', mse_series_final, save_path='results/figures/mse_comparison_final.png')
plot_metric('mae', mae_series_final, save_path='results/figures/mae_comparison_final.png')

## Optional — save results back into the repo

Writes `results/runs.csv` fresh from everything computed above and stages
it plus the figures. Commit/push are left commented out on purpose --
review `git status`/`git diff` first, then uncomment when you're ready.

In [ ]:
RUNS_CSV_HEADER = ['run_id','timestamp','model','dataset','horizon','seq_len','seg_len',
                    'd_model','seed','flags','mse','mae','mase','epoch_time_s','params',
                    'peak_mem_mb','notes']
ts = datetime.datetime.now().isoformat(timespec='seconds')
rows = []

for h in HORIZONS:
    mse, mae = recon[h]
    rows.append([f'SegRNN_ETTh1_{h}_{ts}', ts, 'SegRNN', 'ETTh1', h, 720, 24, 512, 2024,
                 'seg_len=24;d_model=512', mse, mae, '', '', '', '', 'reconstruction'])

for h in HORIZONS:
    for name in ['naive', 'seasonal_naive']:
        b = baselines[h][name]
        rows.append([f'{name}_ETTh1_{h}_{ts}', ts, name, 'ETTh1', h, 720, '', '', 2024,
                     'period=24;features=M;seq_len=720', b['mse'], b['mae'], b['mase'],
                     '', '', '', 'deterministic baseline, no training'])

for h in HORIZONS:
    mse, mae = improved[h]
    rows.append([f'SegRNNTime_ETTh1_{h}_{ts}', ts, 'SegRNNTime', 'ETTh1', h, 720, 24, 512, 2024,
                 'seg_len=24;d_model=512;mark_dim=4', mse, mae, '', '', '', '',
                 'improved: calendar features in encoder'])

with open('results/runs.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(RUNS_CSV_HEADER)
    writer.writerows(rows)

print(f'wrote {len(rows)} rows to results/runs.csv')

!git add results/runs.csv results/figures/
!git status
# review the diff above, then when ready:
# !git commit -m "Update results: reconstruction, baselines, improved model"
# !git push origin main